# Imports

In [10]:
from pathlib import Path
from transformers import AutoTokenizer
import os
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
import gradio as gr
import json, re
from IPython.display import IFrame
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import io
from IPython.display import display as nb_display, Image as NBImage
from pyvis.network import Network
import ipywidgets as widgets

In [13]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

# Paths

In [14]:
ROOT = Path('.').resolve().parents[0]  #Path(r'C:\Users\Archit\Documents\ML Projects\D-RAGon_Psyche')
DATA_DIR = ROOT/'Data'
BOOKS_DIR = ROOT/'Data'/'Raw_Books'
CHROMA_DIR = ROOT/'Chroma'

# Doc ingestion

In [15]:
a = [f for f in os.listdir(BOOKS_DIR) if f.endswith(".pdf")]
a

['Emotional Intelligence — Daniel Goleman.pdf',
 'Full Catastrophe Living — Jon Kabat-Zinn.pdf',
 'Industrial and Organizational Psychology — Paul Levy pdf.pdf',
 'Influence The Psychology of Persuasion — Robert Cialdini .pdf',
 'Leadership in Organizations — Gary Yukl.pdf',
 'Occupational Health Psychology — Stavroula Leka & Jonathan Houdmont.pdf',
 'Organizational Behavior — Robbins & Judge.pdf',
 'Pre-Suasion — Robert Cialdini.pdf',
 'The Burnout Epidemic — Jennifer Moss.pdf',
 'The Fearless Organization — Amy Edmondson.pdf',
 'The Oxford Handbook of Organizational Climate and Culture.pdf',
 'The Relaxation Response — Herbert Benson.pdf',
 'The Stress Management Handbook — Eva Selhub.pdf',
 'The Stress Proof Brain — Melanie Greenberg.pdf',
 'Thinking, Fast and Slow — Daniel Kahneman.pdf',
 'Why Zebras Don’t Get Ulcers — Robert Sapolsky.pdf',
 'Work and Organizational Psychology — John Arnold.pdf']

In [16]:
def clean_book_title(filename):
    name = filename.replace(".pdf", "")
    clean = name.split("—")[0].strip()
    return clean

In [17]:
def load_docs():
    documents = []
    pdf_files = list(BOOKS_DIR.glob("*.pdf"))
    for pdf_path in tqdm(pdf_files, desc="Loading PDFs"):
        try:
            loader = PyPDFLoader(str(pdf_path))
            docs = loader.load()
            clean_title = clean_book_title(pdf_path.name)
            # Attach book-level metadata
            for doc in docs:
                doc.metadata["source_book"] = clean_title
                doc.metadata["full_filename"] = pdf_path.name
                
            documents.extend(docs)
        except Exception as e:
            print(f"Error loading {pdf_path.name}: {e}")
            
    print(f"\nLoaded {len(documents)} pages from {len(pdf_files)} books.")
    return documents

# Chunking

## Page lvl filtering - Removing Boilerplate

In [18]:
def filter_pages(docs, min_chars = 200):
    cleaned = []
    blacklist = [
        "all rights reserved",
        "isbn",
        "table of contents"
    ]
    for d in docs:
        text = d.page_content.strip()
        lower_txt = text.lower()

        if len(text)<min_chars: #removes short pages
            continue
        if any(b in lower_txt for b in blacklist): # removes boilerplate pages
            continue
        cleaned.append(d)
    print(f"Filtered {len(docs) - len(cleaned)} pages. \nRemaining: {len(cleaned)}")
    return cleaned

In [19]:
def token_length(text: str):
    return len(tokenizer.encode(text, add_special_tokens=False))

In [20]:
def split_docs(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,          # tokens  800
        chunk_overlap=300,        # tokens  200
        length_function=token_length,)
    
    chunks = splitter.split_documents(docs)
    # Drop tiny chunks
    chunks = [c for c in chunks if token_length(c.page_content) > 200]
    return chunks

## Logging

In [21]:
def log_corpus_stats(docs, chunks):
    total_tokens = sum(token_length(d.page_content) for d in docs)
    total_chunks = len(chunks)

    source_books = set(d.metadata.get("source_book", "unknown") for d in docs)

    print("\n=== Corpus Statistics ===")
    print(f"Books: {len(source_books)}")
    print(f"Pages: {len(docs)}")
    print(f"Total Tokens: {total_tokens}")
    print(f"Vector Chunks: {total_chunks}")
    print("=========================\n")

## Calculating chunk IDs

In [22]:
def calc_chunk_ids(chunks):
    # Creates unique IDs in the format:
    # source_book:page_number:chunk_index
    last_page_id = None
    current_chunk_index = 0

    for chunk in chunks:
        source_book = chunk.metadata.get("source_book", "unknown")
        page = chunk.metadata.get("page", 0)
        current_page_id = f"{source_book}:{page}"

        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0

        chunk_id = f"{current_page_id}:{current_chunk_index}"
        chunk.metadata["id"] = chunk_id

        last_page_id = current_page_id

    return chunks

# Embeddings Setup

In [23]:
def get_embeddings_function(device: str = None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    return HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True,
                      "batch_size": 16}, # Number of chunks encoded per GPU forward pass. Safe-16, Aggr- 32
    )

# Vector DB

In [24]:
BATCH_SIZE = 32  # Outer Batch size : Number of documents sent to Chroma per insertion step. Safe-32, Aggressive - 64
def add_to_chroma(chunks: list[Document]):
    # load DB
    db = Chroma(persist_directory=CHROMA_DIR,
                embedding_function = get_embeddings_function())

    #Calc Page IDs 
    chunks_with_ids = calc_chunk_ids(chunks)

    # Add or update the documents
    existing_items = db.get(include=[])
    existing_ids = set(existing_items['ids'])
    print(f"Number of existing dicuments in DB: {len(existing_ids)}")

    # Only add docs that don't exist in the DB
    new_chunks = []
    for chunk in chunks_with_ids:
        if chunk.metadata['id'] not in existing_ids:
            new_chunks.append(chunk)
    
    if len(new_chunks):
        print(f"Adding new documents: {len(new_chunks)}")
    
        for i in tqdm(range(0, len(new_chunks), BATCH_SIZE)):
            batch_chunks = new_chunks[i:i+BATCH_SIZE]
            batch_ids = [chunk.metadata['id'] for chunk in batch_chunks]
            db.add_documents(batch_chunks, ids=batch_ids)
            torch.cuda.empty_cache()
        print("Final DB count:", db._collection.count())
    else:
        print("No New Documents to add")

## To Delete DB ⚠️⚠️⚠️

In [25]:
import shutil
import os
def clear_database():
    if os.path.exists(CHROMA_DIR):
        shutil.rmtree(CHROMA_DIR)

# Ingestion

In [26]:
# docs = load_docs()
# docs = filter_pages(docs)
# chunks = split_docs(docs)
# emb_fxn = get_embeddings_function()
# log_corpus_stats(docs, chunks)

In [27]:
# add_to_chroma(chunks)
# db = Chroma(persist_directory=CHROMA_DIR,
#             embedding_function=emb_fxn)

# Model & Retrieval Stack Initialization

In [28]:
emb_fxn = get_embeddings_function()

db = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=emb_fxn
)

llm = OllamaLLM(model="llama3.1")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

# Retriever Setup

## Prompts

In [29]:
Base_Prompt_Recommend2 = """
You are an organizational psychologist providing evidence-based recommendations for workplace stress and wellbeing.

You are speaking directly to the person experiencing the situation. Write your recommendations using second-person language (e.g., "you").

Use only the information provided in the context below. Ground each recommendation in ideas, strategies, or findings mentioned in the context.

Context:
{context}

User's situation:
{question}

Instructions:
- Provide exactly 3 concise recommendations tailored to the user's situation.
- Address the user directly using "you".
- Each recommendation should clearly reflect ideas from the context.
- After each recommendation, briefly reference the relevant idea from the context in parentheses.
- Avoid generic advice unless it is explicitly supported by the context.
- Do not introduce information that is not present in the context.

If the context does not contain enough information to give recommendations, respond with: "I do not know based on the provided context."
"""

## Query Funciton

In [30]:
def query_rag(query_text: str, return_context=False, return_sources=False):
    results = db.similarity_search_with_score(query_text, k=4)
    #reranked_docs = rerank(query_text, results)
    top_docs = [doc for doc, _ in results]

    context = "\n\n---\n\n".join([doc.page_content for doc in top_docs])

    prompt_template = ChatPromptTemplate.from_template(Base_Prompt_Recommend2)
    
    prompt = prompt_template.format(context=context, question=query_text)
    #print(prompt)
    
    response_text = llm.invoke(prompt)
    sources = [doc.metadata.get('id', None) for doc in top_docs]
    
    formatted_response = f'Response: {response_text}\n\nSources: {sources}'
    #print(formatted_response)    
    
    if return_context and return_sources:
        return response_text, context, sources
    if return_sources:
        return response_text, sources
    if return_context:
        return response_text, context
    return response_text

# Controller Layer -- Guided Conversation Flow

Terminal Mode (non-Gradio)  
 Use this to run the pipeline directly in the notebook without the UI  
 conv_controller() → query_rag() for quick testing  

In [31]:
def conv_controller():
    questions = [
    "What's been going on at work lately that's been bothering you?",
    "How has this been affecting you — things like your energy, motivation, or how you feel at the end of the day?",
    "Do you feel the amount of work you're given is manageable, or does it feel overwhelming?",
    "How would you describe the environment in your team — supportive, stressful, or something else?"]
    answers = []  
    for q in questions:
        print("Q: ",q)
        ans = input("A: ")
        answers.append(ans)
        
    qa_pairs = ""
    for q, a in zip(questions, answers):
        qa_pairs += f"Q: {q}\nA: {a}\n\n"
    
    PROMPT_Ctrl = f'''Take the following questions and summarize them into a single paragraph sounding like a case of organizational psychology
{qa_pairs}'''
    
    ctrl_op = llm.invoke(PROMPT_Ctrl).strip()
    return ctrl_op

# Conversational Mode

In [32]:
CHAT_HISTORY = []
def reset_chat_history():
    CHAT_HISTORY.clear()
    print("Chat history reset.")

In [33]:
REWRITE_PROMPT = """
Given the chat history and the latest question, rewrite the question so it is standalone and can be understood without the history.

Chat history:
{history}

Latest question: {question}

Standalone question:
"""

In [34]:
def rewrite_query_with_history(query: str, history: list):
    if not history:
        return query

    # taken upto most recent 10 turns 
    history_text = "\n".join(f"{role}: {msg}" for role, msg in history[-10:])

    prompt = REWRITE_PROMPT.format(history=history_text,question=query,)
    #print(prompt)
    
    rewritten = llm.invoke(prompt)
    return rewritten.strip()

In [35]:
CONV_PROMPT = """
You are an organizational psychologist having a supportive conversation with an employee about their workplace situation.

You have already provided initial recommendations. Now respond to their follow-up naturally and helpfully.

Use only the context below to ground your response. If they push back, offer alternatives from the context. If they ask to elaborate, go deeper. If they ask how to implement something, give practical steps.

Context:
{context}

Conversation so far:
{history}

Follow-up: {question}

Respond conversationally in 2-3 sentences. Stay grounded in the context. Do not give new recommendations unless explicitly asked.
"""

In [36]:
def query_rag_hist(query_text: str, history: list, return_context=False, return_sources=False):
    standalone_query = rewrite_query_with_history(query_text, history)
    
    results = db.similarity_search_with_score(standalone_query, k=10)
    context = "\n\n---\n\n".join([doc.page_content for doc, score in results])
    
    history_text = "\n".join(f"{role}: {msg}" for role, msg in history[-10:])
    prompt_template = ChatPromptTemplate.from_template(CONV_PROMPT)
    prompt = prompt_template.format(context=context, question=query_text, history=history_text,)
    #print(prompt)

    response_text = llm.invoke(prompt)
    
    history = history + [("user", query_text), ("assistant", response_text)]  # no mutation
    sources = [doc.metadata.get('id', None) for doc, score in results]

    #print("Standalone query:", standalone_query)
    #print("\nResponse: ",response_text)
    #print("\nSources:", sources)
    if return_context and return_sources:
        return response_text, context, sources, history
    
    if return_sources:
        return response_text, sources, history
    
    if return_context:
        return response_text, context, history
    
    return response_text, history

# Knowledge Architecture: Framework–Parameter Mapping

In [37]:
def build_knowledge_graph(output_path="knowledge_graph.html"):

    # --- Data ---
    QUESTIONS = [
        ("Q1", "What's been going on\nat work lately?"),
        ("Q2", "How has this affected\nyour energy & motivation?"),
        ("Q3", "Is your workload\nmanageable?"),
        ("Q4", "How would you describe\nyour team environment?"),
    ]
    FRAMEWORKS = [
        ("JD-R", "Job Demands-Resources\n(JD-R) Model"),
        ("MBI",  "Maslach Burnout\nInventory (MBI)"),
        ("KAHN", "Kahn's Employee\nEngagement Model"),
    ]
    PARAMETERS = [
        ("P1",  "Job Demands"),
        ("P2",  "Workload"),
        ("P3",  "Social Support"),
        ("P4",  "Health Impairment Process"),
        ("P5",  "Emotional Exhaustion"),
        ("P6",  "Depersonalization /\nCynicism"),
        ("P7",  "Personal Accomplishment"),
        ("P8",  "Psychological Safety"),
        ("P9",  "Psychological Availability"),
        ("P10", "Meaningfulness"),
        ("P11", "Physical & Emotional\nEngagement"),
    ]
    LITERATURE = [
        ("L1", "The Burnout Epidemic\n- Jennifer Moss"),
        ("L2", "Occupational Health Psychology\n- Leka & Houdmont"),
        ("L3", "The Fearless Organization\n- Amy Edmondson"),
        ("L4", "Organizational Behavior\n- Robbins & Judge"),
        ("L5", "Emotional Intelligence\n- Daniel Goleman"),
    ]

    # --- Edges ---
    Q_to_F = [
        ("Q1","JD-R"),("Q1","MBI"),
        ("Q2","MBI"), ("Q2","JD-R"),
        ("Q3","JD-R"),("Q3","MBI"),
        ("Q4","KAHN"),("Q4","JD-R"),
    ]
    F_to_P = [
        ("JD-R","P1"),("JD-R","P2"),("JD-R","P3"),("JD-R","P4"),
        ("MBI","P2"), ("MBI","P5"), ("MBI","P6"), ("MBI","P7"),
        ("KAHN","P8"),("KAHN","P9"),("KAHN","P10"),("KAHN","P11"),
    ]
    P_to_L = [
        ("P1","L2"),("P2","L1"),("P2","L2"),("P3","L4"),("P4","L2"),
        ("P5","L1"),("P6","L1"),("P7","L4"),("P8","L3"),
        ("P9","L3"),("P10","L4"),("P11","L5"),
    ]

    # --- Colours ---
    COLORS = {
        "question":  {"background":"#3B82F6","border":"#1D4ED8"},
        "framework": {"background":"#10B981","border":"#047857"},
        "parameter": {"background":"#F59E0B","border":"#B45309"},
        "literature":{"background":"#8B5CF6","border":"#5B21B6"},
    }

    net = Network(
        height="720px", width="100%",
        bgcolor="#0f1117", font_color="#e0e0e0",
        directed=True, notebook=True, cdn_resources="in_line"
    )

    def add_nodes(nodes, node_type, shape, size, level):
        for nid, label in nodes:
            net.add_node(nid, label=label, color=COLORS[node_type],
                         shape=shape, size=size, level=level,
                         font={"size":11,"face":"Arial","color":"#f0f0f0"},
                         title=f"<b>{node_type.title()}</b><br>{label.replace(chr(10),' ')}")

    add_nodes(QUESTIONS,  "question",   "ellipse", 28, level=0)
    add_nodes(FRAMEWORKS, "framework",  "box",     24, level=1)
    add_nodes(PARAMETERS, "parameter",  "dot",     18, level=2)
    add_nodes(LITERATURE, "literature", "diamond", 20, level=3)

    def add_edges(edges, color):
        for src, dst in edges:
            net.add_edge(src, dst, color=color, width=1.8, arrows="to",
                         smooth={"type":"curvedCW","roundness":0.15})

    add_edges(Q_to_F, "#3B82F6")
    add_edges(F_to_P, "#10B981")
    add_edges(P_to_L, "#F59E0B")

    # --- Hierarchical layout (neural-net style, L to R) ---
    net.set_options(json.dumps({
        "layout": {
            "hierarchical": {
                "enabled": True,
                "direction": "LR",
                "sortMethod": "directed",
                "levelSeparation": 220,
                "nodeSpacing": 80,
                "treeSpacing": 120
            }
        },
        "physics": {"enabled": False},
        "interaction": {"hover": True, "navigationButtons": True, "tooltipDelay": 100}
    }))

    # Write with utf-8 to avoid Windows cp1252 error
    with open(output_path, "w", encoding="utf-8") as f:
        legend_html = """
        <div style="position:fixed; top:16px; left:16px; background:#1a1a2e;
             border:1px solid #444; border-radius:8px; padding:10px 14px; z-index:999;
             font-family:Arial; font-size:13px; line-height:2;">
          <span style="color:#3B82F6;">&#9679; Questions</span><br>
          <span style="color:#10B981;">&#9632; Frameworks</span><br>
          <span style="color:#F59E0B;">&#9679; Parameters</span><br>
          <span style="color:#8B5CF6;">&#9670; Literature</span>
        </div>
        """
        
        html = net.generate_html()
        html = html.replace("</body>", legend_html + "</body>")
        f.write(html)

    return IFrame(output_path, width="100%", height="720px")

# Parameter Scoring

In [38]:
PARAMETERS = [
    "Job Demands",
    "Workload",
    "Social Support",
    "Health Impairment Process",
    "Emotional Exhaustion",
    "Depersonalization / Cynicism",
    "Personal Accomplishment",
    "Psychological Safety",
    "Psychological Availability",
    "Meaningfulness",
    "Physical & Emotional Engagement",
]

In [39]:
# Colour by framework
PARAM_COLORS = {
    "Job Demands":                  "#3B82F6",  # JD-R blue
    "Workload":                     "#3B82F6",
    "Social Support":               "#3B82F6",
    "Health Impairment Process":    "#3B82F6",
    "Emotional Exhaustion":         "#10B981",  # MBI green
    "Depersonalization / Cynicism": "#10B981",
    "Personal Accomplishment":      "#10B981",
    "Psychological Safety":         "#F59E0B",  # Kahn amber
    "Psychological Availability":   "#F59E0B",
    "Meaningfulness":               "#F59E0B",
    "Physical & Emotional Engagement": "#F59E0B",
}

In [40]:
SCORE_PROMPT = """You are an organizational psychologist. 
Given the case summary below, score each psychological parameter from 0 to 10.
For stress/negative parameters (Job Demands, Workload, Health Impairment Process, 
Emotional Exhaustion, Depersonalization / Cynicism): 
  10 = severely elevated, 0 = not present.
For protective/positive parameters (Social Support, Personal Accomplishment, 
Psychological Safety, Psychological Availability, Meaningfulness, 
Physical & Emotional Engagement): 
  10 = very high/healthy, 0 = completely absent.

Case summary:
{narrative}

Respond ONLY with a valid JSON object. No explanation, no markdown, no extra text.
Keys must exactly match this list:
["Job Demands", "Workload", "Social Support", "Health Impairment Process",
 "Emotional Exhaustion", "Depersonalization / Cynicism", "Personal Accomplishment",
 "Psychological Safety", "Psychological Availability", "Meaningfulness",
 "Physical & Emotional Engagement"]
Values must be integers 0-10."""

In [41]:
#matplotlib.use('Agg')
def score_parameters(narrative, llm):
    # --- Get scores from LLM ---
    prompt = SCORE_PROMPT.format(narrative=narrative)
    raw = llm.invoke(prompt).strip()

    # Strip markdown fences if present
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        scores = json.loads(raw)
    except json.JSONDecodeError:
        # Fallback: try to extract JSON object from response
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        scores = json.loads(match.group()) if match else {p: 0 for p in PARAMETERS}

    # Ensure all keys exist
    scores = {p: int(scores.get(p, 0)) for p in PARAMETERS}

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 5.5))
    fig.patch.set_facecolor("#0f1117")
    ax.set_facecolor("#0f1117")

    params  = list(scores.keys())
    values  = list(scores.values())
    colors  = [PARAM_COLORS[p] for p in params]

    bars = ax.barh(params, values, color=colors, height=0.6, edgecolor="none")

    # Score labels
    for bar, val in zip(bars, values):
        ax.text(val + 0.15, bar.get_y() + bar.get_height() / 2,
                str(val), va="center", ha="left",
                color="#f0f0f0", fontsize=9, fontweight="bold")

    ax.set_xlim(0, 11)
    ax.set_xlabel("Score (0–10)", color="#aaaaaa", fontsize=9)
    ax.set_title("Psychological Parameter Scores", color="#f0f0f0",
                 fontsize=12, fontweight="bold", pad=12)
    ax.tick_params(colors="#cccccc", labelsize=8)
    ax.spines[:].set_visible(False)
    ax.xaxis.set_tick_params(color="#444")
    ax.set_axisbelow(True)
    ax.xaxis.grid(True, color="#222", linewidth=0.6)

    # Legend
    legend_items = [
        mpatches.Patch(color="#3B82F6", label="JD-R"),
        mpatches.Patch(color="#10B981", label="MBI"),
        mpatches.Patch(color="#F59E0B", label="Kahn"),
    ]
    ax.legend(handles=legend_items, loc="lower right",
              facecolor="#1a1a2e", edgecolor="#444",
              labelcolor="#f0f0f0", fontsize=8)

    plt.tight_layout()
    
    # --- Display ---
    buf = io.BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight",
                facecolor=fig.get_facecolor(), dpi=150)
    plt.close()
    buf.seek(0)
    with graph_out:
        graph_out.clear_output(wait=True)
        nb_display(NBImage(buf.read()))
        nb_display(build_knowledge_graph())

    return scores

# Gradio UI

In [42]:
questions = [
    "What's been going on at work lately that's been bothering you?",
    "How has this been affecting you — things like your energy, motivation, or how you feel at the end of the day?",
    "Do you feel the amount of work you're given is manageable, or does it feel overwhelming?",
    "How would you describe the environment in your team — supportive, stressful, or something else?"
]
def handle_input(user_input, chat_history, answers, q_index, mode, narrative, history):
    if mode == "intake":
        return process_intake(user_input, chat_history, answers, q_index, narrative, history)
    else:
        return process_followup(user_input, chat_history, narrative, history)

In [43]:
def process_intake(user_input, chat_history, answers, q_index, narrative, history):
    torch.cuda.empty_cache()
    answers = answers + [user_input]
    chat_history = chat_history + [{"role": "user", "content": user_input}]
    q_index = q_index + 1

    if q_index < len(questions):
        chat_history = chat_history + [{"role": "assistant", "content": questions[q_index]}]
        return chat_history, answers, q_index, "intake", narrative, history, gr.update(placeholder="Type your answer here...", value="")

    else:
        chat_history = chat_history + [{"role": "assistant", "content": "🔍 Analysing your situation..."}]

        qa_pairs = ""
        for q, a in zip(questions, answers):
            qa_pairs += f"Q: {q}\nA: {a}\n\n"

        PROMPT_Ctrl = f"Take the following questions and summarize them into a single paragraph sounding like a case of organizational psychology\n{qa_pairs}"
        narrative = llm.invoke(PROMPT_Ctrl).strip()
        scores = score_parameters(narrative, llm)  # renders chart in notebook

        response, sources = query_rag(narrative, return_sources=True)

        unique_books = list(set([s.split(":")[0] for s in sources]))
        sources_text = "\n".join([f"📚 {book}" for book in unique_books])
        final_response = f"{response}\n\n---\n**Sources:**\n{sources_text}"

        chat_history[-1] = {"role": "assistant", "content": final_response}

        # Seed conversation history for follow-up
        history = [("user", narrative), ("assistant", final_response)]  # not just assistant

        return chat_history, answers, q_index, "chat", narrative, history, gr.update(placeholder="Ask a follow-up question...", value="")

In [44]:
def process_followup(user_input, chat_history, narrative, history):
    chat_history = chat_history + [{"role": "user", "content": user_input}]
    chat_history = chat_history + [{"role": "assistant", "content": "🔍 Thinking..."}]
    
    response, sources, history = query_rag_hist(user_input, history, return_sources=True)
    
    unique_books = list(set([s.split(":")[0] for s in sources]))
    sources_text = "\n".join([f"📚 {book}" for book in unique_books])
    final_response = f"{response}\n\n---\n**Sources:**\n{sources_text}"
    
    chat_history[-1] = {"role": "assistant", "content": final_response}
    return chat_history, [], 4, "chat", narrative, history, gr.update(placeholder="Ask a follow-up question...", value="")

In [45]:
def reset_all():
    return (
        [{"role": "assistant", "content": questions[0]}],
        [], 0, "intake", "", [],
        gr.update(placeholder="Type your answer here...", value="")
    )

In [46]:
graph_out = widgets.Output()
nb_display(graph_out)
# ---- UI ----
with gr.Blocks() as demo:
    chat_history_state = gr.State([])
    answers_state = gr.State([])
    q_index_state = gr.State(0)
    mode_state = gr.State("intake")
    narrative_state = gr.State("")
    history_state = gr.State([])

    gr.Markdown(
        """
        <div style="text-align: center;">
        
        # 🐉 D-RAGon_Psyche
        Evidence-based workplace wellbeing recommendations, *grounded in organizational psychology literature.*
        
        </div>
        """
        )

    chatbot = gr.Chatbot(
        value=[{"role": "assistant", "content": questions[0]}],
        height=500,
        show_label=False
    )

    with gr.Row():
        txt = gr.Textbox(placeholder="Type your answer here...", show_label=False, scale=4, container=False,value = "")
        submit_btn = gr.Button("→", scale=1, variant="primary")

    restart_btn = gr.Button("🔄 Start Over", variant="secondary")

    all_outputs = [chatbot, answers_state, q_index_state, mode_state, narrative_state, history_state, txt]

    submit_btn.click(fn=handle_input,
        inputs=[txt, chatbot, answers_state, q_index_state, mode_state, narrative_state, history_state],
        outputs=all_outputs)

    txt.submit(fn=handle_input,
        inputs=[txt, chatbot, answers_state, q_index_state, mode_state, narrative_state, history_state],
        outputs=all_outputs)

    restart_btn.click(fn=reset_all, outputs=all_outputs)

demo.launch(inline=False, inbrowser=True)

Output()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Knowledge Architecture: Framework–Parameter Mapping

In [47]:
nb_display(build_knowledge_graph())